<a href="https://colab.research.google.com/github/raki-rankawat/stm32-thesis/blob/main/Model_KD_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
# =====================================================
# Imports
# =====================================================

import os
import time
import random
import numpy as np
from pathlib import Path
from PIL import Image
import tarfile
from urllib.request import urlretrieve

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from google.colab import drive

In [68]:
# =====================================================
# Mount Google Drive
# =====================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [69]:
# =====================================================
# Reproducibility + Device Setup
# =====================================================

SEED = 41

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [70]:
# =====================================================
# KD Experiment Config
# =====================================================

KD_MODE = "scratch"          # "ft" or "scratch"
KD_TEMPERATURE = 4.0
KD_ALPHA = 0.7
KD_EPOCHS = 30
KD_LR = 1e-3

SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"

mode_tag = "ft" if KD_MODE == "ft" else "scratch"
temp_tag = str(KD_TEMPERATURE).replace(".", "p")
alpha_tag = str(KD_ALPHA).replace(".", "p")

KD_SAVE_PATH = f"{SAVE_DIR}/mobilenetv2_kd_{mode_tag}_T{temp_tag}_A{alpha_tag}_best.pth"

print("========== KD CONFIG ==========")
print("Mode        :", KD_MODE)
print("Temperature :", KD_TEMPERATURE)
print("Alpha       :", KD_ALPHA)
print("Epochs      :", KD_EPOCHS)
print("LR          :", KD_LR)
print("Save Path   :", KD_SAVE_PATH)

========== KD CONFIG ==========
Mode        : scratch
Temperature : 4.0
Alpha       : 0.7
Epochs      : 30
LR          : 0.001
Save Path   : /content/drive/My Drive/Colab Notebooks/mobilenetv2_kd_scratch_T4p0_A0p7_best.pth


In [71]:
# =====================================================
# Fixed Manifest Paths
# =====================================================

MANIFEST_BASE_DIR = Path("/content/drive/My Drive/vww_fixed_split_manifests")

def manifest_paths():
    return {
        "train_person": MANIFEST_BASE_DIR / "train_person.txt",
        "val_person": MANIFEST_BASE_DIR / "val_person.txt",
        "train_non_person": MANIFEST_BASE_DIR / "train_non_person.txt",
        "val_non_person": MANIFEST_BASE_DIR / "val_non_person.txt",
    }

paths = manifest_paths()

for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"❌ Missing manifest: {name} -> {path}")

In [72]:
# =====================================================
# Manifest Helpers
# =====================================================

def load_manifest(manifest_path):
    with open(manifest_path, "r") as f:
        return [Path(line.strip()) for line in f if line.strip()]

In [73]:
# =====================================================
# Custom Dataset from Manifest
# =====================================================

class VWWManifestDataset(Dataset):
    def __init__(self, person_manifest, nonperson_manifest, transform=None):
        self.transform = transform
        self.samples = []

        person_files = load_manifest(person_manifest)
        nonperson_files = load_manifest(nonperson_manifest)

        # Fixed class mapping
        # non_person -> 0
        # person     -> 1
        for p in nonperson_files:
            self.samples.append((p, 0))

        for p in person_files:
            self.samples.append((p, 1))

        self.class_to_idx = {"non_person": 0, "person": 1}
        self.classes = ["non_person", "person"]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [74]:
# =====================================================
# Dataset Restore (needed because manifests point to /content paths)
# =====================================================

VWW_URL = "https://www.silabs.com/public/files/github/machine_learning/benchmarks/datasets/vw_coco2014_96.tar.gz"

BASE_DIR = Path("/content/vww_work")
ARCHIVE_PATH = BASE_DIR / "vw_coco2014_96.tar.gz"
EXTRACT_DIR = BASE_DIR / "extracted"

def download_vww():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if ARCHIVE_PATH.exists() and ARCHIVE_PATH.stat().st_size > 0:
        print("✅ VWW archive already downloaded")
        return

    print("⬇️ Downloading VWW archive...")
    urlretrieve(VWW_URL, ARCHIVE_PATH)
    print("✅ Download complete:", ARCHIVE_PATH)

def extract_vww():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    if any(EXTRACT_DIR.iterdir()):
        print("✅ VWW already extracted")
        return

    print("📦 Extracting VWW archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(EXTRACT_DIR)
    print("✅ Extraction complete:", EXTRACT_DIR)

def find_vww_root():
    for p in EXTRACT_DIR.rglob("person"):
        if p.is_dir() and (p.parent / "non_person").is_dir():
            return p.parent
    raise RuntimeError("❌ Could not find dataset directories")

download_vww()
extract_vww()
vww_root = find_vww_root()
print("✅ Found VWW root:", vww_root)

✅ VWW archive already downloaded
✅ VWW already extracted
✅ Found VWW root: /content/vww_work/extracted/vw_coco2014_96


In [75]:
# =====================================================
# Data Loaders
# =====================================================

BATCH_SIZE = 64
IMG_SIZE = 96
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

train_data = VWWManifestDataset(
    person_manifest=paths["train_person"],
    nonperson_manifest=paths["train_non_person"],
    transform=train_transform
)

val_data = VWWManifestDataset(
    person_manifest=paths["val_person"],
    nonperson_manifest=paths["val_non_person"],
    transform=eval_transform
)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Class mapping:", train_data.class_to_idx)
print("Train size:", len(train_data))
print("Validation size:", len(val_data))

Class mapping: {'non_person': 0, 'person': 1}
Train size: 8000
Validation size: 2000


In [76]:
# =====================================================
# MobileNetV2 Components
# =====================================================

class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()

        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(
                hidden_dim,
                hidden_dim,
                3,
                stride=stride,
                padding=1,
                groups=hidden_dim,
                bias=False
            ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        return self.block(x)

In [77]:
# =====================================================
# MobileNetV2 Model (Student)
# =====================================================

class VWW_MobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()

        self.initial = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        self.features = nn.Sequential(
            InvertedResidual(32, 16, 1, 1),

            InvertedResidual(16, 24, 2, 6),
            InvertedResidual(24, 24, 1, 6),

            InvertedResidual(24, 32, 2, 6),
            InvertedResidual(32, 32, 1, 6),

            InvertedResidual(32, 64, 2, 6),
            InvertedResidual(64, 64, 1, 6),
        )

        self.final_conv = nn.Sequential(
            nn.Conv2d(64, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU6(inplace=True)
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(512, 2)
        self._initialize_weights()

    def forward(self, x):
        x = self.initial(x)
        x = self.features(x)
        x = self.final_conv(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

In [78]:
# =====================================================
# VGG-Style CNN (Teacher)
# =====================================================

class VWW_VGGStyle(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 : 96 -> 48
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 2 : 48 -> 24
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 3 : 24 -> 12
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 4 : 12 -> 6
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(128, 2)
        )

        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

In [79]:
# =====================================================
# Load Teacher and Student
# =====================================================

TEACHER_PATH = "/content/drive/My Drive/Colab Notebooks/vggstyle_seed_52.pth"
STUDENT_BASELINE_PATH = "/content/drive/My Drive/Colab Notebooks/mobilenetv2_seed_85.pth"

teacher_model = VWW_VGGStyle().to(device)
teacher_model.load_state_dict(torch.load(TEACHER_PATH, map_location=device))
teacher_model.eval()

for param in teacher_model.parameters():
    param.requires_grad = False

student_model = VWW_MobileNetV2().to(device)

if KD_MODE == "ft":
    student_model.load_state_dict(torch.load(STUDENT_BASELINE_PATH, map_location=device))
    print("✅ Student initialized from baseline MobileNet checkpoint")
elif KD_MODE == "scratch":
    print("✅ Student initialized from scratch")
else:
    raise ValueError(f"Invalid KD_MODE: {KD_MODE}. Use 'ft' or 'scratch'.")

print("✅ Teacher and student models loaded successfully")

✅ Student initialized from scratch
✅ Teacher and student models loaded successfully


In [80]:
# =====================================================
# Parameter Count
# =====================================================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Student Params:", count_parameters(student_model))
print("Teacher Params:", count_parameters(teacher_model))

Student Params: 151874
Teacher Params: 0


In [81]:
# =====================================================
# Accuracy Evaluation
# =====================================================

def evaluate(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            outputs = model(X)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [82]:
# =====================================================
# Knowledge Distillation Loss
# =====================================================

class DistillationLoss(nn.Module):
    def __init__(self, temperature=4.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce = nn.CrossEntropyLoss()
        self.kl = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, labels):
        T = self.temperature

        hard_loss = self.ce(student_logits, labels)

        student_log_probs = F.log_softmax(student_logits / T, dim=1)
        teacher_probs = F.softmax(teacher_logits / T, dim=1)

        soft_loss = self.kl(student_log_probs, teacher_probs) * (T * T)

        total_loss = self.alpha * soft_loss + (1.0 - self.alpha) * hard_loss

        return total_loss, hard_loss, soft_loss

In [83]:
# =====================================================
# KD Training Function
# =====================================================

def train_kd(
    student_model,
    teacher_model,
    train_loader,
    val_loader,
    device,
    epochs=30,
    lr=1e-3,
    temperature=4.0,
    alpha=0.7,
    save_path="/content/drive/My Drive/Colab Notebooks/mobilenetv2_kd_best.pth"
):
    optimizer = torch.optim.Adam(student_model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    kd_criterion = DistillationLoss(temperature=temperature, alpha=alpha)

    best_val_acc = evaluate(student_model, val_loader, device)
    print(f"Initial Student Val Accuracy: {best_val_acc*100:.2f}%")
    torch.save(student_model.state_dict(), save_path)
    print(f"✅ Initial student checkpoint saved to: {save_path}")

    start_time = time.time()

    for epoch in range(epochs):
        student_model.train()

        running_total_loss = 0.0
        running_hard_loss = 0.0
        running_soft_loss = 0.0
        correct = 0
        total = 0

        for X, y in train_loader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.no_grad():
                teacher_logits = teacher_model(X)

            student_logits = student_model(X)

            total_loss, hard_loss, soft_loss = kd_criterion(student_logits, teacher_logits, y)

            total_loss.backward()
            optimizer.step()

            batch_size = y.size(0)

            running_total_loss += total_loss.item() * batch_size
            running_hard_loss += hard_loss.item() * batch_size
            running_soft_loss += soft_loss.item() * batch_size

            preds = torch.argmax(student_logits, dim=1)
            correct += (preds == y).sum().item()
            total += batch_size

        scheduler.step()

        train_total_loss = running_total_loss / total
        train_hard_loss = running_hard_loss / total
        train_soft_loss = running_soft_loss / total
        train_acc = correct / total

        val_acc = evaluate(student_model, val_loader, device)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"LR: {scheduler.get_last_lr()[0]:.6f} | "
            f"KD Loss: {train_total_loss:.4f} | "
            f"Hard Loss: {train_hard_loss:.4f} | "
            f"Soft Loss: {train_soft_loss:.4f} | "
            f"Train Acc: {train_acc*100:.2f}% | "
            f"Val Acc: {val_acc*100:.2f}%"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(student_model.state_dict(), save_path)
            print(f"✅ Best KD student saved to: {save_path}")

    total_minutes = (time.time() - start_time) / 60.0

    print("\n========== KD Training Complete ==========")
    print(f"Best Validation Accuracy: {best_val_acc*100:.2f}%")
    print(f"Training Time: {total_minutes:.2f} minutes")

    return best_val_acc, total_minutes

In [84]:
# =====================================================
# Baseline Check Before KD
# =====================================================

baseline_acc = evaluate(student_model, val_loader, device)
teacher_acc = evaluate(teacher_model, val_loader, device)

print("\n========== Before KD ==========")
print(f"Student Baseline Accuracy : {baseline_acc*100:.2f}%")
print(f"Teacher Accuracy          : {teacher_acc*100:.2f}%")


========== Before KD ==========
Student Baseline Accuracy : 50.10%
Teacher Accuracy          : 83.25%


In [85]:
# =====================================================
# Run KD Training
# =====================================================

best_val_acc, kd_training_time = train_kd(
    student_model=student_model,
    teacher_model=teacher_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=KD_EPOCHS,
    lr=KD_LR,
    temperature=KD_TEMPERATURE,
    alpha=KD_ALPHA,
    save_path=KD_SAVE_PATH
)

Initial Student Val Accuracy: 50.10%
✅ Initial student checkpoint saved to: /content/drive/My Drive/Colab Notebooks/mobilenetv2_kd_scratch_T4p0_A0p7_best.pth
Epoch [1/30] | LR: 0.000997 | KD Loss: 1.3055 | Hard Loss: 0.6802 | Soft Loss: 1.5735 | Train Acc: 62.70% | Val Acc: 66.55%
✅ Best KD student saved to: /content/drive/My Drive/Colab Notebooks/mobilenetv2_kd_scratch_T4p0_A0p7_best.pth
Epoch [2/30] | LR: 0.000989 | KD Loss: 1.0614 | Hard Loss: 0.6388 | Soft Loss: 1.2425 | Train Acc: 69.44% | Val Acc: 64.35%
Epoch [3/30] | LR: 0.000976 | KD Loss: 0.9461 | Hard Loss: 0.6180 | Soft Loss: 1.0868 | Train Acc: 72.30% | Val Acc: 72.85%
✅ Best KD student saved to: /content/drive/My Drive/Colab Notebooks/mobilenetv2_kd_scratch_T4p0_A0p7_best.pth
Epoch [4/30] | LR: 0.000957 | KD Loss: 0.8588 | Hard Loss: 0.5807 | Soft Loss: 0.9780 | Train Acc: 73.86% | Val Acc: 73.50%
✅ Best KD student saved to: /content/drive/My Drive/Colab Notebooks/mobilenetv2_kd_scratch_T4p0_A0p7_best.pth
Epoch [5/30] | L

In [86]:
# =====================================================
# Final KD Evaluation
# =====================================================

best_kd_model = VWW_MobileNetV2().to(device)
best_kd_model.load_state_dict(torch.load(KD_SAVE_PATH, map_location=device))

final_kd_acc = evaluate(best_kd_model, val_loader, device)

print("\n========== Final KD Result ==========")
print(f"KD Mode                    : {KD_MODE}")
print(f"Temperature                : {KD_TEMPERATURE}")
print(f"Alpha                      : {KD_ALPHA}")
print(f"Baseline MobileNet Accuracy: {baseline_acc*100:.2f}%")
print(f"Teacher VGG Accuracy       : {teacher_acc*100:.2f}%")
print(f"KD MobileNet Accuracy      : {final_kd_acc*100:.2f}%")
print(f"Improvement vs Baseline    : {(final_kd_acc - baseline_acc)*100:.2f}%")
print(f"Training Time              : {kd_training_time:.2f} min")
print(f"Saved KD Model Path        : {KD_SAVE_PATH}")


========== Final KD Result ==========
KD Mode                    : scratch
Temperature                : 4.0
Alpha                      : 0.7
Baseline MobileNet Accuracy: 50.10%
Teacher VGG Accuracy       : 83.25%
KD MobileNet Accuracy      : 80.50%
Improvement vs Baseline    : 30.40%
Training Time              : 4.84 min
Saved KD Model Path        : /content/drive/My Drive/Colab Notebooks/mobilenetv2_kd_scratch_T4p0_A0p7_best.pth
